# Grokking: Weight Averaging & Distillation ExperimentReproduces the grokking experiment with:- **Baseline**: single model trained on full data- **4 Specialists**: each trained on 25% of the data- **Weight Averaged**: average specialist weights, fine-tune on full data- **Distilled**: knowledge distillation from specialists

In [ ]:
!pip install -q pytorch_lightning mod sympy blobfile numpy pandas matplotlib

## Clone Repo

In [ ]:
import os, sys# Clone this repo - already has validation fix + split_n_ways + distillation metricsREPO_DIR = '/kaggle/working/grok'if not os.path.exists(REPO_DIR):    !git clone https://github.com/yazankb/grok.git {REPO_DIR}sys.path.insert(0, REPO_DIR)os.chdir(REPO_DIR)print(f'Repo at: {REPO_DIR}')

## Imports & GPU

In [ ]:
import torchimport numpy as npimport pandas as pdimport matplotlibmatplotlib.use('Agg')import matplotlib.pyplot as pltfrom grok.multi_training import train_multi_with_distillation, add_multi_argsGPU_ID = 0 if torch.cuda.is_available() else -1DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'print(f'Device: {DEVICE}')

## ConfigurationFor quick test: FINAL_STEPS = 2000

In [ ]:
# Experiment configFINAL_STEPS = 250N_MODELS = 4SPECIALIST_STEPS = FINAL_STEPS // N_MODELSRANDOM_SEED = 42LOGDIR = '/kaggle/working/logs'DATADIR = os.path.join(REPO_DIR, 'data')EXPERIMENT_NAME = 'grokking_experiment'print(f'Final steps: {FINAL_STEPS}')print(f'Specialist steps: {SPECIALIST_STEPS} x {N_MODELS} = {SPECIALIST_STEPS * N_MODELS} total')

## Build Hyperparameters

In [ ]:
parser = add_multi_args()hparams = parser.parse_args([])hparams.logdir = LOGDIRhparams.datadir = DATADIRhparams.math_operator = '/'hparams.train_data_pct = 50hparams.n_models = N_MODELShparams.specialist_steps = SPECIALIST_STEPShparams.final_steps = FINAL_STEPShparams.distill_steps = FINAL_STEPShparams.distill_temperature = 2.0hparams.distill_alpha = 0.5hparams.experiment_name = EXPERIMENT_NAMEhparams.run_baseline = Truehparams.random_seed = RANDOM_SEEDhparams.gpu = GPU_IDhparams.weight_decay = 1hparams.max_lr = 1e-3hparams.warmup_steps = 10hparams.n_layers = 2hparams.n_heads = 4hparams.d_model = 128print(f'Model: {hparams.n_layers}L-{hparams.n_heads}H-{hparams.d_model}D')print(f'Operator: {hparams.math_operator} mod 97 | Train: {hparams.train_data_pct}%')

## Run ExperimentThis runs: 4 specialists -> weight average -> fine-tune -> distill -> baseline

In [ ]:
experiment_dir = train_multi_with_distillation(hparams)print(f'Results: {experiment_dir}')

## Load Metrics

In [ ]:
import osdef find_csv(base_dir, subdir):    path = os.path.join(base_dir, subdir, 'lightning_logs', 'version_0', 'metrics.csv')    return path if os.path.exists(path) else Nonemetrics = {}for i in range(N_MODELS):    p = find_csv(experiment_dir, f'specialist_{i}')    if p: metrics[f'specialist_{i}'] = pfor name in ['merged_average', 'baseline']:    p = find_csv(experiment_dir, name)    if p: metrics[name] = p# Distillation has custom CSVdc = os.path.join(experiment_dir, 'distilled', 'distill_metrics.csv')if os.path.exists(dc): metrics['distilled'] = dcdvc = os.path.join(experiment_dir, 'distilled', 'distill_val_metrics.csv')if os.path.exists(dvc): metrics['distilled_val'] = dvcprint('Found:', list(metrics.keys()))# Load and inspect datadata = {}for k, v in metrics.items():    df = pd.read_csv(v)    # Drop rows where step or target metric is NaN    if 'val_accuracy' in df.columns:        df = df.dropna(subset=['val_accuracy', 'step'])    elif 'student_acc' in df.columns:        df = df.dropna(subset=['student_acc', 'step'])    elif 'full_train_acc' in df.columns:        df = df.dropna(subset=['full_train_acc', 'step'])    data[k] = df    print(f'{k}: {len(df)} rows, cols: {list(df.columns)[:10]}')

## Plot: Accuracy Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))colors = {'baseline': '#2196F3', 'merged_average': '#4CAF50', 'distilled': '#9C27B0'}# Validation Accuracyax = axes[0]for i in range(N_MODELS):    name = f'specialist_{i}'    if name in data and len(data[name]) > 0 and 'val_accuracy' in data[name].columns:        data[name].plot(x='step', y='val_accuracy', ax=ax, alpha=0.3, color='gray', legend=False)for name, color in colors.items():    if name in data and len(data[name]) > 0 and 'val_accuracy' in data[name].columns:        data[name].plot(x='step', y='val_accuracy', ax=ax, label=name, color=color, legend=True)if 'distilled_val' in data and len(data['distilled_val']) > 0:    data['distilled_val'].plot(x='step', y='val_accuracy', ax=ax, label='distilled', color=colors['distilled'], legend=True)ax.set_title('Validation Accuracy')ax.set_xlabel('Step')ax.set_ylabel('Accuracy %')ax.set_ylim(0, 105)ax.grid(True, alpha=0.3)# Training Accuracyax = axes[1]for i in range(N_MODELS):    name = f'specialist_{i}'    if name in data and len(data[name]) > 0:        col = 'full_train_acc' if 'full_train_acc' in data[name].columns else 'train_accuracy'        if col in data[name].columns:            data[name].plot(x='step', y=col, ax=ax, alpha=0.3, color='gray', legend=False)for name, color in colors.items():    if name in data and len(data[name]) > 0:        col = 'full_train_acc' if 'full_train_acc' in data[name].columns else 'train_accuracy'        if col in data[name].columns:            data[name].plot(x='step', y=col, ax=ax, label=name, color=color, legend=True)if 'distilled' in data and len(data['distilled']) > 0 and 'student_acc' in data['distilled'].columns:    data['distilled'].plot(x='step', y='student_acc', ax=ax, label='distilled', color=colors['distilled'], legend=True)ax.set_title('Training Accuracy')ax.set_xlabel('Step')ax.set_ylabel('Accuracy %')ax.set_ylim(0, 105)ax.grid(True, alpha=0.3)plt.tight_layout()plt.savefig('/kaggle/working/accuracy.png', dpi=150)plt.show()

## Plot: Dashboard

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))# Val Accuracyax = axes[0, 0]for name, color in colors.items():    if name in data and len(data[name]) > 0 and 'val_accuracy' in data[name].columns:        data[name].plot(x='step', y='val_accuracy', ax=ax, label=name, color=color)if 'distilled_val' in data and len(data['distilled_val']) > 0:    data['distilled_val'].plot(x='step', y='val_accuracy', ax=ax, label='distilled', color=colors['distilled'])ax.set_title('Validation Accuracy')ax.set_ylim(0, 105)ax.grid(True, alpha=0.3)# Val Loss (log)ax = axes[0, 1]for name, color in colors.items():    if name in data and len(data[name]) > 0 and 'val_loss' in data[name].columns:        data[name].plot(x='step', y='val_loss', ax=ax, label=name, color=color)ax.set_title('Validation Loss')ax.set_yscale('log')ax.grid(True, alpha=0.3)# Train Accuracyax = axes[1, 0]for name, color in colors.items():    if name in data and len(data[name]) > 0:        col = 'full_train_acc' if 'full_train_acc' in data[name].columns else 'train_accuracy'        if col in data[name].columns:            data[name].plot(x='step', y=col, ax=ax, label=name, color=color)if 'distilled' in data and len(data['distilled']) > 0 and 'student_acc' in data['distilled'].columns:    data['distilled'].plot(x='step', y='student_acc', ax=ax, label='distilled', color=colors['distilled'])ax.set_title('Training Accuracy')ax.set_ylim(0, 105)ax.grid(True, alpha=0.3)# Final bar chartax = axes[1, 1]final_acc = []labels = []bar_colors = []for name, label, bar_color in [('baseline', 'Baseline', colors['baseline']), ('merged_average', 'Weight Avg', colors['merged_average']), ('distilled_val', 'Distilled', colors['distilled'])]:    if name in data and len(data[name]) > 0 and 'val_accuracy' in data[name].columns:        acc = float(data[name]['val_accuracy'].iloc[-1])        final_acc.append(acc)        labels.append(label)        bar_colors.append(bar_color)if final_acc:    bars = ax.bar(labels, final_acc, color=bar_colors)    ax.set_ylabel('Accuracy %')    ax.set_title('Final Val Accuracy')    for bar, acc in zip(bars, final_acc):        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1, f'{acc:.1f}%', ha='center')    ax.set_ylim(0, max(final_acc) * 1.2 if final_acc else 100)    ax.grid(True, alpha=0.3, axis='y')plt.suptitle(f'Grokking: {hparams.math_operator} mod 97 | {N_MODELS} specialists | {FINAL_STEPS:,} steps')plt.tight_layout()plt.savefig('/kaggle/working/dashboard.png', dpi=150)plt.show()print('Plots saved to /kaggle/working/')

## Results

In [ ]:
print('='*50)print('FINAL RESULTS')print('='*50)for name, label in [('baseline', 'Baseline'), ('merged_average', 'Weight Avg'), ('distilled_val', 'Distilled')]:    if name in data and len(data[name]) > 0 and 'val_accuracy' in data[name].columns:        acc = float(data[name]['val_accuracy'].iloc[-1])        print(f'{label:15s}: {acc:.1f}%')